# Tests on benchmark dataset

In [21]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [ ]:
import json
import numpy as np
import pandas as pd
from models.metric import CDE, exhaustive_CDE, EF, CDEF
from models.llm_text_embeddings import get_embeddings_for_list

from operator import itemgetter 
from typing import Tuple, Callable

## Load benchmark dataset

In [ ]:
def parse_benchmark(path:str) -> Tuple[list, list, list]:
    
    with open(path) as f:
        dataset = json.load(f)
        dataset = dataset['Data']

    A = []
    B = []
    C = []

    for e in dataset:
        A.append(e['A'])
        B.append(e['B'])
        C.append(e['C'])

    return A,B,C

In [24]:
similarities_path = '../data/external/benchmark/test_similarities.json'
A_sim, B_sim, C_sim = parse_benchmark(similarities_path)

mix_path = '../data/external/benchmark/test_mix.json'
A_mix, B_mix, C_mix = parse_benchmark(mix_path)

types_path = '../data/external/benchmark/test_types.json'
A_type, B_type, C_type = parse_benchmark(types_path)

## Get text embeddings

In [25]:


A_sim_embed = get_embeddings_for_list(A_sim)
B_sim_embed = get_embeddings_for_list(B_sim)
C_sim_embed = get_embeddings_for_list(C_sim)

A_type_embed = get_embeddings_for_list(A_type)
B_type_embed = get_embeddings_for_list(B_type)
C_type_embed = get_embeddings_for_list(C_type)

A_mix_embed = get_embeddings_for_list(A_mix)
B_mix_embed = get_embeddings_for_list(B_mix)
C_mix_embed = get_embeddings_for_list(C_mix)


## Test metric

In [ ]:
def test_metric(
        metric_fun: Callable, 
        A: list, 
        B: list, 
        C: list, 
        beta: float = 1.0, 
        verbose: bool = False
) -> Tuple[int, int, int]:

    epsilon = 1e-9

    n_passed = 0
    n_failed = 0
    ids_failed = []

    for i, (a, b, c) in enumerate(zip(A, B, C)):
        
        if metric_fun is CDEF:
            s1 = metric_fun(a, a, beta)
            s2 = metric_fun(a, b, beta)
            s3 = metric_fun(a, c, beta)
        else:
            s1 = metric_fun(a, a)
            s2 = metric_fun(a, b)
            s3 = metric_fun(a, c)

        if metric_fun is CDE or metric_fun is exhaustive_CDE:
            if (s1 < s2 or abs(s1 - s2) < epsilon) and (s2 < s3 or abs(s2 - s3) < epsilon):
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)
        elif metric_fun is EF:
            if abs(s1) <= abs(s2) <= abs(s3):
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)
        elif metric_fun is CDEF:
            if (s1 > s2 or abs(s1 - s2) < epsilon) and (s2 > s3 or abs(s2 - s3) < epsilon):
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)

    return n_passed, n_failed, ids_failed

In [39]:
def combine_test_results(
        p:int, f:int, id:list, 
        benchmark_name: str, 
        metric_name:str, 
        beta:float = None,
        verbose:bool = True) -> str:

    test_results = {
        "Benchmark": benchmark_name, 
        "Metric": metric_name+"-"+str(beta) if beta is not None else metric_name,
        "Passed": p, 
        "Failed": f, 
        "Type1": (np.array(id) <= 39).sum(),
        "Type2": ((np.array(id) >= 40) & (np.array(id) <= 79)).sum(),
        "Type3": ((np.array(id) >= 80) & (np.array(id) <= 119)).sum(),
        "Type4": (np.array(id) >= 120).sum()
    }
    
    if verbose:
        print(f'{benchmark_name:8s}\t\
{metric_name+"-"+str(beta) if beta is not None else metric_name:10s}\t\
PASSED: {p:3d}\tFAILED: {f:3d}\t\
IDS 0-39: {(np.array(id) <= 39).sum():3d}\t\
IDS 40-79: {((np.array(id) >= 40) & (np.array(id) <= 79)).sum():3d}\t\
IDS 80-119: {((np.array(id) >= 80) & (np.array(id) <= 119)).sum():3d}\t\
IDS 120-129: {(np.array(id) >= 120).sum():3d}')

    return test_results


In [40]:
results = {
    "Benchmark": [],
    "Metric": [],
    "Passed": [],
    "Failed": [],
    "Type1": [],
    "Type2": [],
    "Type3": [],
    "Type4": []
}

benchmark_results = pd.DataFrame(results)

In [41]:
p, f, id = test_metric(CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'SIM', 'CDE')

p, f, id = test_metric(exhaustive_CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'SIM', 'exh_CDE')

p, f, id = test_metric(EF, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'SIM', 'EF')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_sim_embed, B_sim_embed, C_sim_embed, beta=beta, verbose=False)
    benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'SIM', 'CDEF', beta=beta)

    
p, f, id= test_metric(CDEF, A_sim_embed, B_sim_embed, C_sim_embed, beta=0.36, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'SIM', 'CDEF', beta=0.36)

SIM     	CDE       	PASSED: 112	FAILED:  18	IDS 0-39:  11	IDS 40-79:   4	IDS 80-119:   3	IDS 120-129:   0
SIM     	exh_CDE   	PASSED: 112	FAILED:  18	IDS 0-39:  11	IDS 40-79:   4	IDS 80-119:   3	IDS 120-129:   0
SIM     	EF        	PASSED: 121	FAILED:   9	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   9
SIM     	CDEF-0.0  	PASSED: 112	FAILED:  18	IDS 0-39:  11	IDS 40-79:   4	IDS 80-119:   3	IDS 120-129:   0
SIM     	CDEF-0.25 	PASSED: 119	FAILED:  11	IDS 0-39:   4	IDS 40-79:   4	IDS 80-119:   2	IDS 120-129:   1
SIM     	CDEF-0.5  	PASSED: 120	FAILED:  10	IDS 0-39:   0	IDS 40-79:   4	IDS 80-119:   0	IDS 120-129:   6
SIM     	CDEF-0.75 	PASSED: 117	FAILED:  13	IDS 0-39:   0	IDS 40-79:   4	IDS 80-119:   0	IDS 120-129:   9
SIM     	CDEF-1.0  	PASSED: 117	FAILED:  13	IDS 0-39:   0	IDS 40-79:   4	IDS 80-119:   0	IDS 120-129:   9
SIM     	CDEF-1.25 	PASSED: 117	FAILED:  13	IDS 0-39:   0	IDS 40-79:   4	IDS 80-119:   0	IDS 120-129:   9
SIM     	CDEF-1.5  	PASSED: 117	FAILED:  13	ID

In [42]:
p, f, id= test_metric(CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'TYPE', 'CDE')

p, f, id= test_metric(exhaustive_CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'TYPE', 'exh_CDE')

p, f, id= test_metric(EF, A_type_embed, B_type_embed, C_type_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'TYPE', 'EF')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_type_embed, B_type_embed, C_type_embed, beta=beta, verbose=False)
    benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'TYPE', 'CDEF', beta)

TYPE    	CDE       	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	exh_CDE   	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	EF        	PASSED: 121	FAILED:   9	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   9
TYPE    	CDEF-0.0  	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	CDEF-0.25 	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	CDEF-0.5  	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	CDEF-0.75 	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	CDEF-1.0  	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	CDEF-1.25 	PASSED: 130	FAILED:   0	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   0	IDS 120-129:   0
TYPE    	CDEF-1.5  	PASSED: 130	FAILED:   0	ID

In [43]:
p, f, id= test_metric(CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'MIX', 'CDE')

p, f, id= test_metric(exhaustive_CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'MIX', 'exh_CDE')

p, f, id= test_metric(EF, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'MIX', 'EF')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_mix_embed, B_mix_embed, C_mix_embed, beta=beta, verbose=False)
    benchmark_results.loc[len(benchmark_results)] = combine_test_results(p, f, id, 'MIX', 'CDEF', beta)

MIX     	CDE       	PASSED: 108	FAILED:  22	IDS 0-39:  18	IDS 40-79:   3	IDS 80-119:   1	IDS 120-129:   0
MIX     	exh_CDE   	PASSED: 108	FAILED:  22	IDS 0-39:  18	IDS 40-79:   3	IDS 80-119:   1	IDS 120-129:   0
MIX     	EF        	PASSED: 120	FAILED:  10	IDS 0-39:   0	IDS 40-79:   0	IDS 80-119:   4	IDS 120-129:   6
MIX     	CDEF-0.0  	PASSED: 108	FAILED:  22	IDS 0-39:  18	IDS 40-79:   3	IDS 80-119:   1	IDS 120-129:   0
MIX     	CDEF-0.25 	PASSED: 109	FAILED:  21	IDS 0-39:  17	IDS 40-79:   3	IDS 80-119:   1	IDS 120-129:   0
MIX     	CDEF-0.5  	PASSED: 112	FAILED:  18	IDS 0-39:  15	IDS 40-79:   3	IDS 80-119:   0	IDS 120-129:   0
MIX     	CDEF-0.75 	PASSED: 112	FAILED:  18	IDS 0-39:  15	IDS 40-79:   3	IDS 80-119:   0	IDS 120-129:   0
MIX     	CDEF-1.0  	PASSED: 112	FAILED:  18	IDS 0-39:  15	IDS 40-79:   3	IDS 80-119:   0	IDS 120-129:   0
MIX     	CDEF-1.25 	PASSED: 112	FAILED:  18	IDS 0-39:  15	IDS 40-79:   3	IDS 80-119:   0	IDS 120-129:   0
MIX     	CDEF-1.5  	PASSED: 127	FAILED:   3	ID

## Results

In [44]:
benchmark_results

,Benchmark,Metric,Passed,Failed,Type1,Type2,Type3,Type4
0,SIM,CDE,112,18,11,4,3,0
1,SIM,exh_CDE,112,18,11,4,3,0
2,SIM,EF,121,9,0,0,0,9
3,SIM,CDEF-0.0,112,18,11,4,3,0
4,SIM,CDEF-0.25,119,11,4,4,2,1
5,SIM,CDEF-0.5,120,10,0,4,0,6
6,SIM,CDEF-0.75,117,13,0,4,0,9
7,SIM,CDEF-1.0,117,13,0,4,0,9
8,SIM,CDEF-1.25,117,13,0,4,0,9
9,SIM,CDEF-1.5,117,13,0,4,0,9


## Debugging

In [ ]:
def test_metric_debug(metric_fun, A, B, C, beta = 1.0, verbose = False):

    n_passed = 0
    n_failed = 0
    ids_failed = []

    for i, (a, b, c) in enumerate(zip(A, B, C)):
        
        print(f'len a: {len(a)}, len b: {len(b)}, len c: {len(c)}')
        if metric_fun is CDEF:
            s1 = metric_fun(a, a, beta)
            s2 = metric_fun(a, b, beta)
            s3 = metric_fun(a, c, beta)
        else:
            s1 = metric_fun(a, a)
            s2 = metric_fun(a, b)
            s3 = metric_fun(a, c)

        if metric_fun is CDE or metric_fun is exhaustive_CDE:
            if s1 <= s2 <= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)
        elif metric_fun is EF:
            if abs(s1) <= abs(s2) <= abs(s3):
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)
        elif metric_fun is CDEF:
            if s1 >= s2 >= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)

    return n_passed, n_failed, ids_failed

In [34]:
import numpy as np
from itertools import combinations, permutations

def __cosine_similarity(v1:list, v2:list) -> float:
    v1 = np.squeeze(np.asarray(v1))
    v2 = np.squeeze(np.asarray(v2))
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def __cosine_distance(v1:list, v2:list) -> float:
    return 1-__cosine_similarity(v1, v2)

def __cosine_distance_of_embeddings(a:list, b:list) -> float:
    
    if a[1] != b[1]: return 2.0

    return __cosine_distance(a[0], b[0])

def __find_min_dist(x):
    k = x.argmin()
    ncol = x.shape[1]
    return int(k/ncol), int(k%ncol)

def CDE(gold_embeddings:list, generated_embeddings:list) -> float:

    min_len = min(len(gold_embeddings), len(generated_embeddings))

    distance_matrix = np.array(
        [[ __cosine_distance_of_embeddings(i,j) for i in gold_embeddings] for j in generated_embeddings]
        )

    cumulative_distance = 0.0

    while distance_matrix.any():
        max_i, max_j = __find_min_dist(distance_matrix)
        cumulative_distance += distance_matrix[max_i, max_j]
        distance_matrix = np.delete(distance_matrix, max_i, axis=0)
        distance_matrix = np.delete(distance_matrix, max_j, axis=1)

    return cumulative_distance/min_len

def exhaustive_CDE(gold_embeddings:list, generated_embeddings:list) -> float:

    if len(gold_embeddings) < len(generated_embeddings):
        min_len = len(gold_embeddings)
        short_list = gold_embeddings
        long_list = generated_embeddings
    else: 
        min_len = len(generated_embeddings)
        short_list = generated_embeddings
        long_list = gold_embeddings

    minimal_average_distance = 2.0

    for combination in list(combinations(long_list, min_len)):
        for permutation in list(permutations(combination)):
            cumulative_distance = 0.0

            for a, b in zip(permutation, short_list):
                cumulative_distance += __cosine_distance_of_embeddings(a, b)

            minimal_average_distance = min(
                minimal_average_distance, cumulative_distance/min_len)
    
    return minimal_average_distance

def EF(gold_embeddings:list, generated_embeddings:list) -> float:
    a = len(gold_embeddings)
    b = len(generated_embeddings)
    return 2*b/(a+b) - 1

def CDEF(gold_embeddings:list, generated_embeddings:list, beta:float=1) -> float:
    a = 1-CDE(gold_embeddings, generated_embeddings)/2
    b = 1-abs(EF(gold_embeddings, generated_embeddings))
    return (1+beta*beta)*a*b/(beta*beta*a + b)





In [35]:
b = list(range(40,80,1))
b = [53,57,65,68]
p, f, id= test_metric_debug(
    CDE, 
    itemgetter(*b)(A_sim_embed), 
    itemgetter(*b)(B_sim_embed), 
    itemgetter(*b)(C_sim_embed), 
    beta=0.0, verbose=True)

len a: 1, len b: 1, len c: 1
TEST FAILED	-2.220446049250313e-16	0.13148049985205845	0.11530888463442845

len a: 1, len b: 1, len c: 1
TEST FAILED	-2.220446049250313e-16	0.17267196944274033	0.07478120331620797

len a: 1, len b: 1, len c: 1
TEST FAILED	0.0	0.17617744147400538	0.03851139259400771

len a: 1, len b: 1, len c: 1
TEST FAILED	2.220446049250313e-16	0.20781361268334697	0.07398375096331156



In [36]:
b = [120, 121, 122, 123,124, 125, 126, 127, 128, 129]
p, f, id= test_metric_debug(
    CDE, 
    itemgetter(*b)(A_type_embed), 
    itemgetter(*b)(B_type_embed), 
    itemgetter(*b)(C_type_embed), 
    beta=0.0, verbose=True)

len a: 1, len b: 1, len c: 1
TEST PASSED	1.1102230246251565e-16	2.0	2.0

len a: 2, len b: 1, len c: 2
TEST PASSED	0.0	2.0	2.0

len a: 1, len b: 2, len c: 1
TEST PASSED	1.1102230246251565e-16	0.6788647669377313	2.0

len a: 2, len b: 1, len c: 2
TEST PASSED	0.0	2.0	2.0

len a: 1, len b: 2, len c: 1
TEST PASSED	0.0	0.6388722263761621	2.0

len a: 2, len b: 1, len c: 2
TEST PASSED	-1.1102230246251565e-16	2.0	2.0

len a: 2, len b: 1, len c: 2
TEST PASSED	1.1102230246251565e-16	2.0	2.0

len a: 1, len b: 2, len c: 1
TEST PASSED	2.220446049250313e-16	0.619585541038758	2.0

len a: 2, len b: 1, len c: 2
TEST PASSED	0.0	2.0	2.0

len a: 2, len b: 1, len c: 2
TEST PASSED	-1.1102230246251565e-16	2.0	2.0



In [37]:
# b = [120,121,122,123,124,125,126,127,128,129]
# b = [128,129]
# p, f, id= test_metric_debug(
#     exhaustive_CDE, 
#     itemgetter(*b)(A_type_embed), 
#     itemgetter(*b)(B_type_embed), 
#     itemgetter(*b)(C_type_embed), 
#     beta=0.0, verbose=False)

b = [90,92]
p, f, id= test_metric_debug(
    CDEF, 
    itemgetter(*b)(A_sim_embed), 
    itemgetter(*b)(B_sim_embed), 
    itemgetter(*b)(C_sim_embed), 
    beta=0.0, verbose=False)


print(f'SIM\tCDEF-0\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id= test_metric_debug(
    CDE, 
    itemgetter(*b)(A_sim_embed), 
    itemgetter(*b)(B_sim_embed), 
    itemgetter(*b)(C_sim_embed), 
    beta=0.0, verbose=False)


print(f'SIM\tCDE\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

len a: 2, len b: 2, len c: 3
len a: 2, len b: 2, len c: 3
SIM	CDEF-0	PASSED: 0	FAILED: 2	IDS FAIL: [0, 1]
len a: 2, len b: 2, len c: 3
len a: 2, len b: 2, len c: 3
SIM	CDE	PASSED: 2	FAILED: 0	IDS FAIL: []
